In [1]:
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import chromadb

load_dotenv(override=True)

client = chromadb.HttpClient(host="localhost", port=8000)
collection = client.get_collection(name="heartopia_knowledge")

model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\ccass\Documents\2026\Github\LLM-Zoomcamp\Main project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5906.02it/s]


In [ ]:
def build_rag_context(user_query: str, top_k: int = 6):
    query_vector = model.encode(user_query).tolist()

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k
    )

    retrieved_facts = results['documents'][0]
    full_context = "\n".join(retrieved_facts)

    llm_prompt = f"""You are an expert Heartopia game assistant and financial advisor. 
Use ONLY the following Context to answer the User's question. 

You are highly encouraged to perform step-by-step mathematical calculations using the prices provided in the Context to find total profits or costs. 

Do not hallucinate prices, ingredients, or locations. If the base items requested by the user are not contained within the Context below, respond with "I am only a financial advisor, I cannot answer that question based on my current data."

Context:
{full_context}

User Question: {user_query}
"""
    return llm_prompt

In [3]:
from google import genai
from openai import OpenAI

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

gemini_client = genai.Client() 
openai_client = OpenAI(api_key=OPENAI_API_KEY)

In [6]:
def generate_rag_response(prompt_text):
    """
    Attempts to generate an answer with Gemini 2.5 Flash first. 
    If a rate limit or error occurs, it falls back to OpenAI GPT-4o-mini.
    """
    try:
        response = gemini_client.models.generate_content(
            model='gemini-3.5-flash',
            contents=prompt_text
        )
        return response.text
        
    except Exception as gemini_error:
        print(f"{gemini_error}")
        try:
            openai_response = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "You are a helpful Heartopia game assistant and financial advisor."},
                    {"role": "user", "content": prompt_text}
                ]
            )
            return openai_response.choices[0].message.content
            
        except Exception as openai_error:
            return f"OpenAI Error: {openai_error}"

In [7]:
query = "I have 5 3-star Great green macaw bird cards and 2 4-star Tiramisu, how much do I get from these?"

print(f"User Query: {query}")
print("-" * 50)

final_prompt = build_rag_context(query, top_k=3)

final_answer = generate_rag_response(final_prompt)

print(final_answer)

User Query: I have 5 3-star Great green macaw bird cards and 2 4-star Tiramisu, how much do I get from these?
--------------------------------------------------
I am only a financial advisor, I cannot answer that question based on my current data.
